# Crash Course — R과 Bioconductor 빠른 시작

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/r-bioconductor.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/r-bioconductor.md)

## 최소 실행 예시

### 예시 1 — 이름으로 맞추고, 유전자마다 검정하고, BH 보정

In [ ]:
set.seed(11)
genes <- sprintf("g%03d", 1:200)
samples <- c("S4", "S2", "S6", "S1", "S5", "S3")                 # 장비가 내보낸 열 순서
counts <- matrix(rnbinom(1200, mu = 300, size = 100), nrow = 200, dimnames = list(genes, samples))
counts[1:5, c("S4", "S5", "S6")] <- counts[1:5, c("S4", "S5", "S6")] * 3   # 앞 5개 유전자만 처리군에서 증가
meta <- data.frame(group = factor(rep(c("ctrl", "treat"), each = 3), levels = c("ctrl", "treat")),
                   row.names = paste0("S", 1:6))
counts <- counts[, rownames(meta)]                                 # 이름으로 정렬
stopifnot(identical(colnames(counts), rownames(meta)))
logcpm <- log2(t(t(counts) / colSums(counts)) * 1e6 + 1)
p <- apply(logcpm, 1, function(v) t.test(v ~ meta$group)$p.value)
lfc <- rowMeans(logcpm[, meta$group == "treat"]) - rowMeans(logcpm[, meta$group == "ctrl"])
res <- data.frame(log2FC = round(lfc, 2), p = signif(p, 3), padj = signif(p.adjust(p, "BH"), 3))
head(res[order(res$p), ], 6)
table(BH_0.05 = res$padj < 0.05)

### 예시 2 — factor 기준 수준이 부호를 정한다

In [ ]:
expr  <- c(7.9, 8.1, 8.0, 9.2, 9.0, 9.4)                      # 한 유전자의 log2 값(가상)
geno  <- c("WT", "WT", "WT", "KO", "KO", "KO")
g1 <- factor(geno)                                            # 기본: 알파벳순 -> KO가 기준
g2 <- factor(geno, levels = c("WT", "KO"))                    # WT를 기준으로 명시
coef(lm(expr ~ g1))
coef(lm(expr ~ g2))